# Bab 14. Regresi Linear dan Gradient Descent

Kode pendamping buku *Python untuk Machine Learning dan Data
Science*. Jalankan selnya berurutan dari atas, sebab sebagian
sel memakai peubah dari sel sebelumnya.

Notebook ini dibangkitkan dari naskah buku. Jangan disunting di
sini, sunting listing pada berkas `.tex` lalu bangkitkan ulang.

## 1. Jawaban acuan

In [ ]:
import numpy as np

luas  = np.array([36., 45., 54., 60., 70., 80., 90., 100.])
kamar = np.array([1., 2., 2., 3., 3., 3., 4., 4.])
y = np.array([326.2, 391.4, 424.6, 469.9,
              515.4, 553.1, 628.5, 680.7])

X = np.column_stack([luas, kamar])
n = len(y)

mu, sd = X.mean(axis=0), X.std(axis=0)
Z  = (X - mu) / sd
Zb = np.column_stack([np.ones(n), Z])

w_acuan = np.linalg.solve(Zb.T @ Zb, Zb.T @ y)
print(np.round(w_acuan, 6))

Keluaran yang diharapkan:

```
[498.725     92.611268  20.38678 ]
```

## 2. Menghitung batas kestabilan

In [ ]:
H = (2/n) * Zb.T @ Zb
lam = np.linalg.eigvalsh(H)
print(np.round(lam, 4))
print(round(2 / lam.max(), 4))

Keluaran yang diharapkan:

```
[0.1032 2.     3.8968]
0.5132
```

## 3. Pengaruh standardisasi

In [ ]:
Xb = np.column_stack([np.ones(n), X])   # skala asli

for A, nama in [(Xb, "asli"), (Zb, "baku")]:
    Hk = (2/n) * A.T @ A
    lmax = np.linalg.eigvalsh(Hk).max()
    print(nama,
          "cond =", round(np.linalg.cond(A), 2),
          "eta_max =", f"{2/lmax:.3e}")

Keluaran yang diharapkan:

```
asli cond = 257.05 eta_max = 2.035e-04
baku cond = 6.14 eta_max = 5.132e-01
```

## 4. Gradient descent dari nol

In [ ]:
def gradient_descent(A, y, eta=0.5, iters=200):
    n, p = A.shape
    w = np.zeros(p)
    riwayat = np.empty(iters + 1)

    for t in range(iters):
        r = A @ w - y             # residu, bentuk (n,)
        riwayat[t] = (r @ r) / n  # biaya saat ini
        g = (2.0 / n) * (A.T @ r) # gradien, bentuk (p,)
        w = w - eta * g           # langkah menurun

    r = A @ w - y
    riwayat[iters] = (r @ r) / n
    return w, riwayat

w_gd, hist = gradient_descent(Zb, y, eta=0.5, iters=200)
print(np.round(w_gd, 6))
print(round(hist[-1], 6))
print(np.allclose(w_gd, w_acuan, atol=1e-2))

Keluaran yang diharapkan:

```
[498.725     92.60896   20.386272]
26.365968
True
```

## 5. Menerjemahkan ke satuan asli

In [ ]:
b_asli = w_gd[0] - np.sum(w_gd[1:] * mu / sd)
beta   = w_gd[1:] / sd

print(round(b_asli, 4))
print(np.round(beta, 4))

Keluaran yang diharapkan:

```
142.9764
[ 4.4538 21.0549]
```

## 6. Mini-batch gradient descent

In [ ]:
def minibatch_gd(A, y, eta=0.1, epochs=50, B=4, seed=0):
    n, p = A.shape
    rng = np.random.default_rng(seed)
    w = np.zeros(p)

    for _ in range(epochs):
        urutan = rng.permutation(n)
        for awal in range(0, n, B):
            idx = urutan[awal:awal + B]
            Ab, yb = A[idx], y[idx]
            r = Ab @ w - yb
            g = (2.0 / len(idx)) * (Ab.T @ r)
            w = w - eta * g
    return w